In [1]:
from __future__ import annotations

import pandas as pd
import numpy as np
from pathlib import Path
from typing import Sequence
import warnings
warnings.filterwarnings("ignore")

In [2]:
# =============================================================================
# CONFIG
# =============================================================================

DATA_DIR = Path("/home/abhishekh/abhi/external/hcdt/")

HCDT_FILES = {
    "DRUG": {"path": DATA_DIR / "DRUG.tsv", "type": "tsv"},
    "DRUG_GENE": {"path": DATA_DIR / "DRUG-GENE.tsv", "type": "tsv"},
    "DRUG_DISEASE": {"path": DATA_DIR / "Drug-Disease.xlsx", "type": "excel"},
    "DRUG_PATHWAY": {"path": DATA_DIR / "Drug-Pathway.xlsx", "type": "excel"},
    "PATHWAY_GENE": {"path": DATA_DIR / "Pathway-Gene.xlsx", "type": "excel"},
}


In [3]:

# =============================================================================
# HELPERS
# =============================================================================

def _read_table(path: Path, file_type: str) -> pd.DataFrame:
    if file_type == "tsv":
        return pd.read_csv(path, sep="\t")
    if file_type == "excel":
        return pd.read_excel(path)

    if file_type == "csv":
        return pd.read_csv(path)
    raise ValueError(file_type)


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    # df = df.dropna()
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[ \-]+", "_", regex=True)
    )
    return df


def normalize_strings(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.select_dtypes(include=["object"]).columns:
        df[c] = df[c].astype("string").str.strip()
    return df


def enforce_id_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    for c in ["drug_id", "gene_id"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    for c in ["pathway_id", "disease_id"]:
        if c in df.columns:
            df[c] = df[c].astype("string")
    return df


def replace_zero_with_na(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            # df[c] = df[c].replace(0, pd.NA)
            df[c] = df[c].replace([0, "0", 0.0], pd.NA)
    return df


def resolve_id_priority(df: pd.DataFrame, out_col: str, priority_cols: Sequence[str]) -> pd.DataFrame:
    df = df.copy()
    if out_col not in df.columns:
        df[out_col] = pd.NA
    for c in priority_cols:
        if c in df.columns:
            df[out_col] = df[out_col].combine_first(df[c])
    return df


def drop_missing_required(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    cols = [c for c in cols if c in df.columns]
    return df.dropna(subset=cols) if cols else df


def deduplicate(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    cols = [c for c in cols if c in df.columns]
    return df.drop_duplicates(subset=cols) if cols else df


# =============================================================================
# DRUG–GENE
# =============================================================================

In [4]:


drug_gene = _read_table(HCDT_FILES["DRUG_GENE"]["path"], HCDT_FILES["DRUG_GENE"]["type"])
drug_gene = standardize_columns(drug_gene)

drug_gene = drug_gene.rename(columns={
    "pubchem_cid": "drug_id",
    "gene_symbol": "gene_name",
})
drug_gene = normalize_strings(drug_gene)
drug_gene = resolve_id_priority(
    drug_gene,
    "gene_id",
    ["gene_id", "entrez_id", "ensembl_id"]
)

drug_gene = normalize_strings(drug_gene)
drug_gene = enforce_id_dtypes(drug_gene)
drug_gene = drop_missing_required(drug_gene, ["drug_id", "gene_id"])



drug_gene_edges = (
    drug_gene[["drug_id", "gene_id"]]
    .drop_duplicates()
    .sort_values(["drug_id", "gene_id"])
    .reset_index(drop=True)
)

drug_gene_edges.to_parquet("drug_gene_association.parquet", index=False)


drug_gene_edges.head()


,drug_id,gene_id
0,6,367
1,6,1588
2,6,2950
3,6,3417
4,6,3559


In [5]:
# Fail fast if any cell contains ';' in any column
if drug_gene_edges.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    raise ValueError("DataFrame contains ';' in one or more cells.")


In [6]:
print(f"Dimension of drug_gene_edges: {drug_gene_edges.shape}")

Dimension of drug_gene_edges: (1149689, 2)


# =============================================================================
# DERIVED TABLE FROM DRUG-GENE
# =============================================================================


In [7]:
drug_master_from_drug_gene = (
    drug_gene[["drug_id", "drug_name"]]
    .replace({r";.*$": ""}, regex=True)   # keep only text before first ';'
    .dropna()
    .drop_duplicates(subset=["drug_id"])
    .sort_values("drug_id")
    .reset_index(drop=True)
)


# =============================================================================
# DRUG–DISEASE
# =============================================================================

In [8]:
drug_disease = _read_table(HCDT_FILES["DRUG_DISEASE"]["path"], HCDT_FILES["DRUG_DISEASE"]["type"])
drug_disease = standardize_columns(drug_disease)

drug_disease = drug_disease.rename(columns={
    "pubchem_cid": "drug_id",
    "disease_name": "disease_name",
    "mesh": "mesh_id",
})

drug_disease = replace_zero_with_na(drug_disease, ["mesh_id", "icd_11"])

drug_disease = resolve_id_priority(
    drug_disease,
    "disease_id",
    ["mesh_id", "icd_11"]
)

drug_disease = normalize_strings(drug_disease)
drug_disease = enforce_id_dtypes(drug_disease)
drug_disease = drop_missing_required(drug_disease, ["drug_id", "disease_id"])


drug_disease_edges = (
    drug_disease[["drug_id", "disease_id"]]
    .drop_duplicates()
    .sort_values(["drug_id", "disease_id"])
    .reset_index(drop=True)
)

drug_disease_edges.to_parquet("drug_disease_association.parquet", index=False)

drug_disease_edges.head()

,drug_id,disease_id
0,1,D002386
1,1,D009437
2,1,D010146
3,6,4A01-4B41
4,6,D008545


In [9]:
# Fail fast if any cell contains ';' in any column
if drug_disease_edges.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    raise ValueError("DataFrame contains ';' in one or more cells.")


In [10]:
print(f"Dimension of drug_disease_edges: {drug_disease_edges.shape}")

Dimension of drug_disease_edges: (16177, 2)


# =============================================================================
# DERIVED TABLE FROM DRUG–DISEASE
# =============================================================================


In [11]:
drug_master_from_drug_disease = (
    drug_disease[["drug_id", "drug_name"]]
    .dropna()
    .drop_duplicates(subset=["drug_id"])
    .sort_values("drug_id")
    .reset_index(drop=True)
)

In [12]:
# Fail fast if any cell contains ';' in any column
if drug_master_from_drug_disease.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    raise ValueError("DataFrame contains ';' in one or more cells.")



# =============================================================================
# DRUG–PATHWAY
# =============================================================================

In [13]:


drug_pathway = _read_table(HCDT_FILES["DRUG_PATHWAY"]["path"], HCDT_FILES["DRUG_PATHWAY"]["type"])
drug_pathway = standardize_columns(drug_pathway)

drug_pathway = drug_pathway.rename(columns={
    "pubchem_cid": "drug_id",
    "path_name": "pathway_name",
})

drug_pathway = replace_zero_with_na(
    drug_pathway,
    ["pathway_id", "reactome_id", "kegg_hsaid"]
)

drug_pathway = resolve_id_priority(
    drug_pathway,
    "pathway_id",
    ["pathway_id", "reactome_id", "kegg_hsaid"]
)

drug_pathway = normalize_strings(drug_pathway)
drug_pathway = enforce_id_dtypes(drug_pathway)
drug_pathway = drop_missing_required(drug_pathway, ["drug_id", "pathway_id"])


drug_pathway_edges = (
    drug_pathway[["drug_id", "pathway_id"]]
    .drop_duplicates()
    .sort_values(["drug_id", "pathway_id"])
    .reset_index(drop=True)
)

drug_pathway_edges.to_parquet("drug_pathway_association.parquet", index=False)

drug_pathway_edges.head()

,drug_id,pathway_id
0,6,R-ALL-30661
1,6,R-ALL-9605649
2,6,R-ALL-9693326
3,6,R-HSA-1614635
4,6,R-HSA-1643685


In [14]:
# Fail fast if any cell contains ';' in any column
if drug_pathway_edges.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    raise ValueError("DataFrame contains ';' in one or more cells.")


In [15]:
print(f"Dimension of drug_pathway_edges: {drug_pathway_edges.shape}")

Dimension of drug_pathway_edges: (25173, 2)


# ============================================================
# Build Gene master table FROM DRUG-GENE
# ============================================================

In [16]:
gene_master = (
    drug_gene[["gene_id", "gene_name"]]
    .dropna()
    .drop_duplicates(subset=["gene_id"])
    .sort_values("gene_id")
    .reset_index(drop=True)
)

gene_master.to_parquet("gene_master_table.parquet", index=False)

gene_master.head()

,gene_id,gene_name
0,1,A1BG
1,2,A2M
2,9,NAT1
3,10,NAT2
4,12,SERPINA3


In [17]:
# Fail fast if any cell contains ';' in any column
if gene_master.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    raise ValueError("DataFrame contains ';' in one or more cells.")


In [18]:
print(f"Dimension of gene_master: {gene_master.shape}")

Dimension of gene_master: (5636, 2)


# =============================================================================
# DERIVED TABLE FROM DRUG-PATHWAY
# =============================================================================


In [19]:
pathway_master_from_drug_pathway = (
    drug_pathway[["pathway_id", "pathway_name"]]
    .dropna()
    .drop_duplicates(subset=["pathway_id"])
)

drug_master_from_drug_pathway = (
    drug_pathway[["drug_id", "drug_name"]]
    .dropna()
    .drop_duplicates(subset=["drug_id"])
    .sort_values("drug_id")
    .reset_index(drop=True)
)


In [20]:
# Fail fast if any cell contains ';' in any column
if drug_master_from_drug_pathway.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    raise ValueError("DataFrame contains ';' in one or more cells.")


# =============================================================================
# PATHWAY–GENE
# =============================================================================


In [21]:
pathway_gene = _read_table(
    HCDT_FILES["PATHWAY_GENE"]["path"],
    HCDT_FILES["PATHWAY_GENE"]["type"]
)
pathway_gene = standardize_columns(pathway_gene)

pathway_gene = pathway_gene.rename(columns={
    "path_name": "pathway_name",
    "gene_symbol": "gene_name",
})

# Treat placeholder zeros as missing
pathway_gene = replace_zero_with_na(
    pathway_gene,
    ["pathway_id", "reactome_id", "kegg_hsaid"]
)

# Resolve pathway_id deterministically
pathway_gene = resolve_id_priority(
    pathway_gene,
    "pathway_id",
    ["pathway_id", "reactome_id", "kegg_hsaid"]
)

pathway_gene = normalize_strings(pathway_gene)
pathway_gene = drop_missing_required(pathway_gene, ["pathway_id", "gene_name"])

# Explode comma-separated gene symbols
pathway_gene["gene_name"] = pathway_gene["gene_name"].str.split(",")
pathway_gene = pathway_gene.explode("gene_name")
pathway_gene["gene_name"] = pathway_gene["gene_name"].str.strip()

# --- CASE-NORMALIZED MAPPING (correct) ---
gene_master["gene_name_norm"] = gene_master["gene_name"].str.upper()
pathway_gene["gene_name_norm"] = pathway_gene["gene_name"].str.upper()

gene_lookup = (
    gene_master
    .drop_duplicates("gene_name_norm")
    .set_index("gene_name_norm")["gene_id"]
)

pathway_gene["gene_id"] = pathway_gene["gene_name_norm"].map(gene_lookup)

# Keep only resolvable genes
pathway_gene = pathway_gene.dropna(subset=["gene_id"])
pathway_gene["gene_id"] = pathway_gene["gene_id"].astype("Int64")
pathway_gene_full=pathway_gene[["gene_id", "pathway_id", "pathway_name"]].drop_duplicates()
pathway_gene = pathway_gene_full[["gene_id", "pathway_id"]].drop_duplicates()
pathway_gene.reset_index(inplace=True, drop=True)
pathway_gene.to_parquet("pathway_gene_association.parquet", index=False)

pathway_gene.head()


,gene_id,pathway_id
0,2990,R-ALL-158857
1,10941,R-ALL-158857
2,54575,R-ALL-158857
3,54576,R-ALL-158857
4,54577,R-ALL-158857


In [22]:
# Fail fast if any cell contains ';' in any column
if pathway_gene.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    raise ValueError("DataFrame contains ';' in one or more cells.")


In [23]:
print(f"Dimension of pathway_gene: {pathway_gene.shape}")

Dimension of pathway_gene: (49477, 2)


# =============================================================================
# DERIVED TABLE FROM PATHWAY–GENE
# =============================================================================


In [24]:
# pathway_gene

In [25]:
pathway_master_pathway_gene = pathway_gene_full[["pathway_id","pathway_name"]]
pathway_master_pathway_gene.drop_duplicates(inplace=True)
pathway_master_pathway_gene.reset_index(inplace=True, drop=True)
pathway_master_pathway_gene.head()

,pathway_id,pathway_name
0,R-ALL-158857,Pentose and glucuronate interconversions
1,R-ALL-426026,Fatty acid degradation
2,hsa00130,Ubiquinone and other terpenoid-quinone biosynt...
3,R-ALL-9841230,Steroid hormone biosynthesis
4,R-HSA-75109,Arginine biosynthesis


In [26]:
# Fail fast if any cell contains ';' in any column
if pathway_master_pathway_gene.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    raise ValueError("DataFrame contains ';' in one or more cells.")


# =============================================================================
# DRUG MASTER
# =============================================================================

In [27]:


drug_master_raw = _read_table(HCDT_FILES["DRUG"]["path"], HCDT_FILES["DRUG"]["type"])
drug_master_raw = standardize_columns(drug_master_raw)

drug_master_raw = drug_master_raw.rename(columns={
    "pubchem_cid": "drug_id",
    "drug_name": "drug_name",
    "mw": "drug_molecular_weight",
    "mf": "drug_molecular_formula",
})

drug_master_raw = drug_master_raw[
    ["drug_id", "drug_name"]
]

drug_master_raw = normalize_strings(drug_master_raw)
drug_master_raw = enforce_id_dtypes(drug_master_raw)
drug_master_raw = drop_missing_required(drug_master_raw, ["drug_id", "drug_name"])
drug_master_raw = deduplicate(drug_master_raw, ["drug_id"])
drug_master_raw = drug_master_raw.sort_values("drug_id").reset_index(drop=True)

drug_master = pd.concat([drug_master_raw, drug_master_from_drug_pathway, drug_master_from_drug_disease, drug_master_from_drug_gene])
# drug_master.dropna(inplace=True)
# drug_master.drop_duplicates(inplace=True, subset=["drug_id"])
# drug_master.reset_index(inplace=True, drop=True)
drug_master = drug_master.dropna()
drug_master = drug_master.drop_duplicates(subset=["drug_id"])
drug_master = drug_master.reset_index(drop=True)

drug_master = (
    drug_master[["drug_id", "drug_name"]]
    .replace({r";.*$": ""}, regex=True)   # keep only text before first ';'
    .dropna()
    .drop_duplicates(subset=["drug_id"])
    .sort_values("drug_id")
    .reset_index(drop=True)
)

drug_master.to_parquet("drug_master_table.parquet", index=False)

drug_master.head()

,drug_id,drug_name
0,1,Acetylcarnitine
1,6,Dinitrochlorobenzene
2,7,9-Ethyladenine
3,15,PS(18:0/20:1(11Z))
4,19,"2,3-DIHYDROXY-BENZOIC ACID"


In [28]:
# Fail fast if any cell contains ';' in any column
if drug_master.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    raise ValueError("DataFrame contains ';' in one or more cells.")


In [29]:
print(f"Dimension of drug_master: {drug_master.shape}")

Dimension of drug_master: (682766, 2)



# ============================================================
# Build DISEASE master table
# ============================================================

In [30]:
disease_master = (
    drug_disease[["disease_id", "disease_name"]]
    .dropna()
    .drop_duplicates(subset=["disease_id"])
    .sort_values("disease_id")
    .reset_index(drop=True)
)

# keep only text before first ';'
disease_master["disease_name"] = (
    disease_master["disease_name"]
    .astype(str)
    .str.split(";", n=1)
    .str[0]
    .str.strip()
)

# drop empty/blank names created by the split
disease_master = disease_master.replace({"": pd.NA}).dropna(subset=["disease_name"])


disease_master.to_parquet("disease_master_table.parquet", index=False)

disease_master.head()

,disease_id,disease_name
0,1A00,Vibrio cholerae infection
1,1A00-1A09,Methicillin-resistant staphylococci infection
2,1A00-1C4Z,Bacterial infection
3,1A00-CA43.1,Inflammation
4,1A02,Bacillary dysentery


In [31]:
# Fail fast if any cell contains ';' in any column
if disease_master.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    raise ValueError("DataFrame contains ';' in one or more cells.")


In [32]:
print(f"Dimension of disease_master: {disease_master.shape}")

Dimension of disease_master: (1513, 2)



# ============================================================
# Build pathway master table
# ============================================================

In [33]:
pathway_master = pd.concat([pathway_master_pathway_gene, pathway_master_from_drug_pathway])
pathway_master.drop_duplicates(inplace=True, subset=["pathway_id"])
pathway_master.reset_index(inplace=True, drop=True)
pathway_master.to_parquet("pathway_master_table.parquet", index=False)
pathway_master.head()

,pathway_id,pathway_name
0,R-ALL-158857,Pentose and glucuronate interconversions
1,R-ALL-426026,Fatty acid degradation
2,hsa00130,Ubiquinone and other terpenoid-quinone biosynt...
3,R-ALL-9841230,Steroid hormone biosynthesis
4,R-HSA-75109,Arginine biosynthesis


In [34]:
# Fail fast if any cell contains ';' in any column
if pathway_master.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    raise ValueError("DataFrame contains ';' in one or more cells.")


In [35]:
print(f"Dimension of pathway_master: {pathway_master.shape}")

Dimension of pathway_master: (1616, 2)


In [36]:
print('='*80)
print('Highly Confident Drug-Target database SUMMARY')
print('='*80)

print('\n📊 MASTER TABLES:')
print(f'  • drug_master_table: {len(drug_master):,} rows')
print(f'    Columns: {list(drug_master.columns)}')
print(f'\n  • gene_master_table: {len(gene_master):,} rows')
print(f'    Columns: {list(gene_master.columns)}')
print(f'\n  • disease_master_table: {len(disease_master):,} rows')
print(f'    Columns: {list(disease_master.columns)}')
print(f'\n  • pathway_master_table: {len(pathway_master):,} rows')
print(f'    Columns: {list(pathway_master.columns)}')

print('\n🔗 EDGE TABLES:')
print(f'  • drug_gene_association: {len(drug_gene_edges):,} rows')
print(f'    Columns: {list(drug_gene_edges.columns)}')
print(f'\n  • drug_disease_association: {len(drug_disease_edges):,} rows')
print(f'    Columns: {list(drug_disease_edges.columns)}')
print(f'\n  • drug_pathway_association: {len(drug_pathway_edges):,} rows')
print(f'    Columns: {list(drug_pathway_edges.columns)}')
print(f'\n  • pathway_gene_association: {len(pathway_gene):,} rows')
print(f'    Columns: {list(pathway_gene.columns)}')

print(f"\nNumber of unique drug name: {drug_master['drug_name'].nunique() if 'drug_name' in drug_master.columns else 'N/A'}")
print(f"Number of unique gene name: {gene_master['gene_name'].nunique() if 'gene_name' in gene_master.columns else 'N/A'}")
print(f"Number of unique disease name: {disease_master['disease_name'].nunique() if 'disease_name' in disease_master.columns else 'N/A'}")
print(f"Number of unique pathway name: {pathway_master['pathway_name'].nunique() if 'pathway_name' in pathway_master.columns else 'N/A'}")

print('\n' + '='*80)
print('✅ HCDT PREPROCESSING COMPLETE!')
print('='*80)


Highly Confident Drug-Target database SUMMARY

📊 MASTER TABLES:
  • drug_master_table: 682,766 rows
    Columns: ['drug_id', 'drug_name']

  • gene_master_table: 5,636 rows
    Columns: ['gene_id', 'gene_name', 'gene_name_norm']

  • disease_master_table: 1,513 rows
    Columns: ['disease_id', 'disease_name']

  • pathway_master_table: 1,616 rows
    Columns: ['pathway_id', 'pathway_name']

🔗 EDGE TABLES:
  • drug_gene_association: 1,149,689 rows
    Columns: ['drug_id', 'gene_id']

  • drug_disease_association: 16,177 rows
    Columns: ['drug_id', 'disease_id']

  • drug_pathway_association: 25,173 rows
    Columns: ['drug_id', 'pathway_id']

  • pathway_gene_association: 49,477 rows
    Columns: ['gene_id', 'pathway_id']

Number of unique drug name: 678567
Number of unique gene name: 5636
Number of unique disease name: 1298
Number of unique pathway name: 1616

✅ HCDT PREPROCESSING COMPLETE!


In [37]:
# ===============================
# DROP ROWS WITH '' OR '.' IN ANY CELL (FINAL CLEANUP)
# ===============================

def drop_empty_or_dot_rows(df: pd.DataFrame) -> pd.DataFrame:
    mask = df.apply(lambda col: col.astype(str).isin(["", "."]), axis=0).any(axis=1)
    return df.loc[~mask].reset_index(drop=True)

# Apply to final tables

drug_master = drop_empty_or_dot_rows(drug_master)
gene_master = drop_empty_or_dot_rows(gene_master)
disease_master = drop_empty_or_dot_rows(disease_master)
pathway_master = drop_empty_or_dot_rows(pathway_master)

drug_gene_edges = drop_empty_or_dot_rows(drug_gene_edges)
drug_disease_edges = drop_empty_or_dot_rows(drug_disease_edges)
drug_pathway_edges = drop_empty_or_dot_rows(drug_pathway_edges)
pathway_gene = drop_empty_or_dot_rows(pathway_gene)

# Re-save cleaned outputs

drug_gene_edges.to_parquet("drug_gene_association.parquet", index=False)
drug_disease_edges.to_parquet("drug_disease_association.parquet", index=False)
drug_pathway_edges.to_parquet("drug_pathway_association.parquet", index=False)

gene_master.to_parquet("gene_master_table.parquet", index=False)
pathway_gene.to_parquet("pathway_gene_association.parquet", index=False)

drug_master.to_parquet("drug_master_table.parquet", index=False)
disease_master.to_parquet("disease_master_table.parquet", index=False)
pathway_master.to_parquet("pathway_master_table.parquet", index=False)
